# Exploratory Data Analysis on Books Dataset

**Author:** Data Analysis Project  
**Date:** June 2026  

## Purpose

This notebook analyzes a dataset of 1000 books scraped from an online bookstore. The goal is to answer three questions:
1. How are book prices distributed, and are there meaningful pricing tiers?
2. Do ratings differ across price segments?
3. Which categories dominate the catalog, and how do they compare on price and rating?

**Note on data quality artifacts:** Two category labels — `Default` (152 books) and `Add a comment` (67 books) — appear to be scraping artifacts rather than real genres. These are flagged throughout the analysis and excluded from category-level comparisons.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import f_oneway

print("EDA Environment Ready!")

EDA Environment Ready!


In [2]:
df = pd.read_csv("../data/books_full.csv")
df.head()

,No.,Title,Category,Price (£),Rating,Availability
0,1,It's Only the Himalayas,Travel,45.17,2,In stock
1,2,Full Moon over Noah’s Ark: An Odyssey to Mount...,Travel,49.43,4,In stock
2,3,See America: A Celebration of Our National Par...,Travel,48.87,3,In stock
3,4,Vagabonding: An Uncommon Guide to the Art of L...,Travel,36.94,2,In stock
4,5,Under the Tuscan Sun,Travel,37.33,3,In stock


# 1. Data Quality Assessment

Before any analysis, I'll check for nulls, duplicates, and any columns that may carry no analytical value.

In [3]:
print(f"Shape: {df.shape}")
print()
df.info()

Shape: (1000, 6)

<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 6 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   No.           1000 non-null   int64  
 1   Title         1000 non-null   str    
 2   Category      1000 non-null   str    
 3   Price (£)     1000 non-null   float64
 4   Rating        1000 non-null   int64  
 5   Availability  1000 non-null   str    
dtypes: float64(1), int64(2), str(3)
memory usage: 47.0 KB


### Insight

The dataset has 1000 rows and 6 columns: `No.`, `Title`, `Category`, `Price (£)`, `Rating`, and `Availability`. Three are numeric and three are string/object.

In [4]:
df.isnull().sum()

No.             0
Title           0
Category        0
Price (£)       0
Rating          0
Availability    0
dtype: int64

### Insight

The dataset is complete — no nulls anywhere. No imputation strategy is needed, and all 1000 records can be used as-is.

In [5]:
print("Availability column value counts:")
print(df['Availability'].value_counts())
print()
print(f"Unique values: {df['Availability'].nunique()}")

Availability column value counts:
Availability
In stock    1000
Name: count, dtype: int64

Unique values: 1


### Insight

The `Availability` column is 100% 'In stock' across all 1000 rows — it carries zero analytical information. This is a data quality limitation: either the scrape only captured in-stock books, or stock status wasn't recorded properly. This column will be excluded from further analysis.

In [6]:
print(f"Fully duplicated rows: {df.duplicated().sum()}")
print(f"Duplicate titles: {df['Title'].duplicated().sum()}")
print()
print("The duplicated title:")
print(df[df['Title'].duplicated(keep=False)][['Title', 'Category', 'Price (£)', 'Rating']])

Fully duplicated rows: 0
Duplicate titles: 1

The duplicated title:
                      Title Category  Price (£)  Rating
700  The Star-Touched Queen  Fantasy      46.02       5
707  The Star-Touched Queen  Fantasy      32.30       5


### Insight

There are no fully duplicated rows. However, one title — *The Star-Touched Queen* — appears twice, both in Fantasy, but at different prices (£46.02 vs £32.30) and different row numbers. This is most likely two different editions or a pricing update captured at different scrape times. Since the records are not identical and both may represent real listings, **both entries are kept** — but this is flagged as a minor data quality note.

# 2. Feature Engineering

I'll create two derived features to support the analysis: a price segment label and a readable rating label.

In [7]:
# Prices range from £10.00 to £59.99 — bins cover the full range
bins = [0, 20, 40, 60]
labels = ['Low (£0–20)', 'Medium (£20–40)', 'Premium (£40–60)']

df['Price_Segment'] = pd.cut(df['Price (£)'], bins=bins, labels=labels)

print(df['Price_Segment'].value_counts())

Price_Segment
Premium (£40–60)    403
Medium (£20–40)     401
Low (£0–20)         196
Name: count, dtype: int64


### Insight

40.3% of books are premium-priced (£40–£60), and Medium and Premium segments are nearly equal in size (401 vs 403). Only 196 books (19.6%) fall in the Low tier. This suggests the scrape skews toward higher-end titles, or the site simply does not carry many cheap books.

In [8]:
rating_map = {1: 'Poor', 2: 'Fair', 3: 'Average', 4: 'Good', 5: 'Excellent'}
df['Rating_Label'] = df['Rating'].map(rating_map)

print("Rating distribution:")
print(df['Rating_Label'].value_counts())
print()
print("Percentage:")
print(round(df['Rating_Label'].value_counts(normalize=True) * 100, 1))

Rating distribution:
Rating_Label
Poor         226
Average      203
Fair         196
Excellent    196
Good         179
Name: count, dtype: int64

Percentage:
Rating_Label
Poor         22.6
Average      20.3
Fair         19.6
Excellent    19.6
Good         17.9
Name: proportion, dtype: float64


### Insight

Ratings are spread fairly evenly across all five levels, with Poor (22.6%) being the most common and Good (17.9%) the least. There's no strong skew toward high or low ratings — the distribution is roughly uniform, which is unusual for a real bookstore where ratings often cluster high.

# 3. Univariate Analysis

I'll look at each variable individually to spot patterns, distributions, and potential issues.

In [9]:
print(df['Price (£)'].describe())
print()
print(f"Skewness: {df['Price (£)'].skew():.4f}")

count    1000.00000
mean       35.07035
std        14.44669
min        10.00000
25%        22.10750
50%        35.98000
75%        47.45750
max        59.99000
Name: Price (£), dtype: float64

Skewness: -0.0375


### Insight

Prices range from £10.00 to £59.99, with a mean of £35.07 and median of £35.98. A skewness of −0.04 confirms an almost perfectly symmetrical price distribution .... the mean and median are close not by coincidence, but because prices are genuinely balanced around the center of the range.

In [10]:
print("All category counts:")
print(df['Category'].value_counts())
print()
print("Top 10 as percentage:")
print(round(df['Category'].value_counts(normalize=True).head(10) * 100, 2))

All category counts:
Category
Default               152
Nonfiction            110
Sequential Art         75
Add a comment          67
Fiction                65
Young Adult            54
Fantasy                48
Romance                35
Mystery                32
Food and Drink         30
Childrens              29
Historical Fiction     26
Classics               19
Poetry                 19
History                18
Womens Fiction         17
Horror                 17
Science Fiction        16
Science                14
Music                  13
Business               12
Travel                 11
Philosophy             11
Thriller               11
Humor                  10
Autobiography           9
Art                     8
Religion                7
Psychology              7
New Adult               6
Christian Fiction       6
Spirituality            6
Sports and Games        5
Biography               5
Self Help               5
Health                  4
Contemporary            3
Christia

### Insight

The two largest 'categories' are scraping artifacts: `Default` (152 books, 15.2%) and `Add a comment` (67 books, 6.7%) are not real genres ...... they appear to be UI elements captured by the scraper. Together, they account for 219 books (21.9% of the dataset). These will be excluded from category-level comparisons. Among real genres, Nonfiction (110), Sequential Art (75), Fiction (65), and Young Adult (54) lead.

# 4. Price Outlier Detection

Using the IQR method to check for unusual pricing.

In [11]:
Q1 = df['Price (£)'].quantile(0.25)
Q3 = df['Price (£)'].quantile(0.75)
IQR = Q3 - Q1
lower = Q1 - (1.5 * IQR)
upper = Q3 + (1.5 * IQR)

print(f"IQR bounds: lower = £{lower:.2f}, upper = £{upper:.2f}")
outliers = df[(df['Price (£)'] < lower) | (df['Price (£)'] > upper)]
print(f"Outliers found: {len(outliers)}")
outliers

IQR bounds: lower = £-15.92, upper = £85.48
Outliers found: 0


,No.,Title,Category,Price (£),Rating,Availability,Price_Segment,Rating_Label


### Insight

No price outliers exist within the IQR bounds. This is consistent with the near-zero skewness observed earlier.....the prices form a clean, compact distribution without extreme values at either end.

# 5. Bivariate Analysis

Now I'll look at how variables relate to each other — specifically price vs rating, and category vs average price.

In [12]:
print("Average price by rating:")
print(df.groupby('Rating')['Price (£)'].mean().round(2))

Average price by rating:
Rating
1    34.56
2    34.81
3    34.69
4    36.09
5    35.37
Name: Price (£), dtype: float64


### Insight

Average prices are nearly identical across all five rating levels (ranging from ~£34.56 to ~£36.09). There's no meaningful relationship between what a book costs and how highly it's rated ... more expensive books are not better rated.

# 6. Multivariate Analysis — Category Summary

Combining price, rating, and volume into a single category-level view. Scraping artifacts (`Default`, `Add a comment`) are excluded here.

In [13]:
# Exclude scraping artifacts
artifacts = ['Default', 'Add a comment']
df_clean = df[~df['Category'].isin(artifacts)].copy()

category_summary = df_clean.groupby('Category').agg(
    Avg_Price=('Price (£)', 'mean'),
    Avg_Rating=('Rating', 'mean'),
    Book_Count=('Title', 'count')
).round(2)

# Show categories with at least 10 books, sorted by average price
print("Categories with 10+ books, sorted by average price:")
category_summary[category_summary['Book_Count'] >= 10].sort_values('Avg_Price', ascending=False)

Categories with 10+ books, sorted by average price:


,Avg_Price,Avg_Rating,Book_Count
Category,,,
Travel,39.79,2.73,11
Fantasy,39.59,3.08,48
History,37.29,2.94,18
Womens Fiction,36.79,3.12,17
Classics,36.55,2.47,19
Fiction,36.07,3.18,65
Poetry,35.97,3.53,19
Horror,35.95,2.71,17
Music,35.64,3.15,13


### Insight

Among real categories with enough books to be meaningful (10+), Travel and Fantasy command the highest average prices (~£39–40), while Food & Drink, Mystery, and Thriller are at the lower end (~£31–32). Notably, Poetry has one of the highest average ratings (3.53) despite mid-range pricing, while Science Fiction has the lowest average rating (2.25) in this filtered set. High-volume categories like Nonfiction (110 books, £34.26) and Fiction (65 books, £36.07) sit close to the overall mean.

# 7. Hypothesis Testing — ANOVA

**H₀:** The mean price is equal across all five rating groups.  
**H₁:** At least one rating group has a significantly different mean price.

A one-way ANOVA will test whether rating level has any effect on price.

In [ ]:
groups = [
    df[df['Rating'] == r]['Price (£)']
    for r in sorted(df['Rating'].unique())
]

f_stat, p_value = f_oneway(*groups)


grand_mean = df['Price (£)'].mean()
ss_between = sum(
    len(g) * (g.mean() - grand_mean)**2 for g in groups
)
ss_total = sum((df['Price (£)'] - grand_mean)**2)
eta_squared = ss_between / ss_total

print(f"F-statistic : {f_stat:.4f}")
print(f"P-value     : {p_value:.4f}")
print(f"η² (eta sq) : {eta_squared:.4f}")

F-statistic : 0.3659
P-value     : 0.8330
η² (eta sq) : 0.0015


### Insight

We fail to reject H₀. With p = 0.833 (well above 0.05), there is no statistically significant difference in mean price across rating groups. Effect size η² ≈ 0.001 confirms this — rating explains less than 0.1% of the variance in price. Rating and price are essentially independent of each other in this dataset.

# 8. Conclusion

Here's what this analysis found:

**Pricing:** Prices are nearly uniformly distributed between £10 and £60, with a skewness of −0.04. Medium (40.1%) and Premium (40.3%) segments are almost equal, with only 19.6% of books in the Low tier. No price outliers exist.

**Ratings:** Ratings are spread roughly evenly across all five levels..unusually flat for a real bookstore. No rating group is strongly dominant.

**Price vs. Rating:** ANOVA confirms no significant relationship (p = 0.833, η² ≈ 0.001). Paying more does not get you a better-rated book.

**Categories:** Two category labels (`Default`, `Add a comment`) are scraping artifacts covering 21.9% of the data — a significant data quality issue. Among real genres, Fantasy and Travel are priciest; Poetry has the best average rating in its size tier.

**Data limitations:** The `Availability` column is uniformly 'In stock' across all rows and carries no analytical value. The one title duplicate (*The Star-Touched Queen*) is retained as it represents different editions.